In [1]:
import sys

sys.path.append("..")

from common.metrics import metrics
from common.preprocessing import preprocessing_cn7
from common.train_test_split import split_data
from common.model_composition import model_composition

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 데이터 불러오기
train_cn7 = pd.read_csv(r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_cn7.csv")

In [3]:
X_cn7, y_cn7 = preprocessing_cn7(train_cn7)

print("X shape:", X_cn7.shape)
print("y shape:", y_cn7.shape)

X shape: (1211, 23)
y shape: (1211,)


In [4]:
import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score


cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

X_array = X_cn7.to_numpy()
y_array = y_cn7.to_numpy()

oof_if_score = np.zeros(len(X_cn7))


for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_array, y_array),
    start=1
):

    X_train = X_array[train_idx]
    y_train = y_array[train_idx]

    X_val = X_array[val_idx]

    # 정상 데이터만 학습
    X_train_normal = X_train[y_train == 0]

    model = IsolationForest(
        n_estimators=500,
        contamination="auto",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_normal)

    # sklearn은 값이 작을수록 이상치
    # 우리가 사용할 점수는 클수록 이상하도록 부호 반전
    oof_if_score[val_idx] = -model.score_samples(X_val)


print(
    "Isolation Forest OOF PR-AUC:",
    average_precision_score(
        y_array,
        oof_if_score
    )
)

Isolation Forest OOF PR-AUC: 0.1802911511580746


In [5]:
if_df = pd.DataFrame({
    "Target": y_array,
    "IsolationForest_Score": oof_if_score
})

if_df["Rank"] = (
    if_df["IsolationForest_Score"]
    .rank(
        ascending=False,
        method="min"
    )
)

display(
    if_df[
        if_df["Target"] == 1
    ].sort_values("Rank")
)

,Target,IsolationForest_Score,Rank
120,1,0.581967,6.0
119,1,0.579783,8.0
117,1,0.578779,9.0
118,1,0.572829,10.0
116,1,0.557751,12.0
115,1,0.555657,13.0
95,1,0.545904,17.0
107,1,0.504456,72.0
111,1,0.495048,109.0
109,1,0.491064,125.0
